### Dataset and Task Metadata

In [93]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="soybean_large",
    dataset_year="1980",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://archive.ics.uci.edu/dataset/90/soybean+large",
    download_description="""

mkdir -p local-data-warehouse/soybean_large/ && wget -P local-data-warehouse/soybean_large/ https://archive.ics.uci.edu/static/public/90/soybean+large.zip && unzip local-data-warehouse/soybean_large/soybean+large.zip -d local-data-warehouse/soybean_large/ && rm local-data-warehouse/soybean_large/soybean+large.zip
""",
    # References
    academic_reference_bibtex="""@article{Michalski1980LearningBB,
  title={Learning by Being Told and Learning from Examples: An Experimental Comparison of the Two Methods of Knowledge Acquisition in the Context of Developing an Expert System for Soybean Disease Diagnosis},
  author={Ryszard S. Michalski and R. L. Chilausky},
  journal={International Journal of Policy Analysis and Information Systems},
  year={1980},
  volume={4},
  number={2}
}
""",
    academic_reference_bibtex_key="Michalski1980LearningBB",
    license="CC BY 4.0",
    data_tags=["IID"],
    curation_comments="""
    - We remove "2-4-d-injury" class as it has only a single sample.
    - We convert all attributes to categorical type.
    - Anomaly: the dataset contains missing values only for four classes ("phytophthora-rot","diaporthe-pod-&-stem-blight", "cyst-nematode", "herbicide-injury").
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="disease",
    problem_type="multiclass_classification",
    objective_metric_name="roc_auc",
    stratify_on="disease",
)

## Preprocessing

In [94]:
import pandas as pd
import numpy as np

column_names = [
"disease",
"date",
"plant-stand",
"precip",
"temp",
"hail",
"crop",
"area",
"severity",
"seed-tmt",
"germination",
"plant-growth",
"leaves",
"leafspots-halo",
"leafspots-marg",
"leafspot-size",
"leaf-shread",
"leaf-malf",
"leaf-mild",
"stem",
"lodging",
"stem-cankers",
"canker-lesion",
"fruiting-bodies",
"external decay",
"mycelium",
"int-discolor",
"sclerotia",
"fruit-pods",
"fruit spots",
"seed",
"mold-growth",
"seed-discolor",
"seed-size",
"shriveling",
"roots",
]

df = pd.read_csv(dataset_mold.path / "soybean-large.data", header=None, index_col=False, names=column_names)
df.replace("?", np.nan, inplace=True)
df = df[df["disease"]!="2-4-d-injury"]
df = df.astype('category')

df = df.sample(frac=1, random_state=99).reset_index(drop=True)

## Data Checks

In [95]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 306
Columns: 36
Use sampling: False (sample size: 306)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['date', 'area', 'canker-lesion', 'stem-cankers', 'severity', 'crop', 'fruit-pods', 'fruit spots', 'leaf-mild', 'fruiting-bodies']
Rows remaining as candidates after top-10 filter: 118 (of 306)

#### Duplicate Report
Total duplicate rows: 4 (1.31% of dataset)
Duplicate rows ignoring target: 4 (1.31% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [96]:
# Sample Rows
df_head

,disease,date,plant-stand,precip,temp,hail,crop,area,severity,seed-tmt,germination,plant-growth,leaves,leafspots-halo,leafspots-marg,leafspot-size,leaf-shread,leaf-malf,leaf-mild,stem,lodging,stem-cankers,canker-lesion,fruiting-bodies,external decay,mycelium,int-discolor,sclerotia,fruit-pods,fruit spots,seed,mold-growth,seed-discolor,seed-size,shriveling,roots
0,alternarialeaf-spot,4,0,2,1,0,2,2,z0,0,1,0,1,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,bacterial-blight,2,0,1,1,0,3,1,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,charcoal-rot,6,0,0,2,0,1,3,1,1,0,1,1,0,2,2,0,0,0,1,0,0,3,0,0,0,2,1,0,4,0,0,0,0,0,0
3,brown-stem-rot,5,0,0,1,0,3,2,1,0,1,0,1,0,2,2,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
4,charcoal-rot,5,0,0,2,1,3,3,1,1,2,1,1,0,2,2,0,0,0,1,0,0,3,0,0,0,2,1,0,4,0,0,0,0,0,0


In [97]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,hail,category,40.0,13.07,2.0,"0, 1"
1,severity,category,40.0,13.07,4.0,"1, 0, 2, z0"
2,seed-tmt,category,40.0,13.07,3.0,"0, 1, 2"
3,lodging,category,40.0,13.07,2.0,"0, 1"
4,germination,category,35.0,11.44,3.0,"1, 2, 0"
5,fruiting-bodies,category,34.0,11.11,3.0,"0, 1, 3"
6,fruit spots,category,34.0,11.11,4.0,"0, 4, 1, 2"
7,seed-discolor,category,34.0,11.11,2.0,"0, 1"
8,shriveling,category,34.0,11.11,2.0,"0, 1"
9,leaf-mild,category,29.0,9.48,3.0,"0, 1, 2"


In [98]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [99]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column          rank                                   
area            1                       1    101  33.01
                2                       3     91  29.74
                3                       2     63  20.59
                4                       0     51  16.67
canker-lesion   1                       0    151  49.35
                2                       2     75  24.51
                3                       1     38  12.42
                4                       3     32  10.46
                5                    <NA>     10   3.27
crop            1                       2     99  32.35
                2                       3     93  30.39
                3                       1     79  25.82
                4                       0     35  11.44
date            1                       5     65  21.24
                2                       4     58  18.95
                3                       3     53  17.32
                4                       2     41  13.40
                5                       6     41  13.40
disease         1     alternarialeaf-spot     40  13.07
                2              brown-spot     40  13.07
                3        phytophthora-rot     40  13.07
                4      frog-eye-leaf-spot     40  13.07
                5          brown-stem-rot     20   6.54
external decay  1                       0    232  75.82
                2                       1     64  20.92
                3                    <NA>     10   3.27
fruit spots     1                       0    164  53.59
                2                       4     54  17.65
                3                    <NA>     34  11.11
                4                       1     29   9.48
                5                       2     25   8.17
fruit-pods      1                       0    193  63.07
                2                       1     53  17.32
                3                       3     30   9.80
                4                    <NA>     24   7.84
                5                       2      6   1.96
fruiting-bodies 1                       0    226  73.86
                2                       1     45  14.71
                3                    <NA>     34  11.11
                4                       3      1   0.33
germination     1                       1     99  32.35
                2                       2     88  28.76
                3                       0     84  27.45
                4                    <NA>     35  11.44
hail            1                       0    211  68.95
                2                       1     55  17.97
                3                    <NA>     40  13.07
int-discolor    1                       0    266  86.93
                2                       1     20   6.54
                3                       2     10   3.27
                4                    <NA>     10   3.27
leaf-malf       1                       0    268  87.58
                2                    <NA>     25   8.17
                3                       1     13   4.25
leaf-mild       1                       0    257  83.99
                2                    <NA>     29   9.48
                3                       1     10   3.27
                4                       2     10   3.27
leaf-shread     1                       0    233  76.14
                2                       1     48  15.69
                3                    <NA>     25   8.17
leafspot-size   1                       1    147  48.04
                2                       2    109  35.62
                3                       0     25   8.17
                4                    <NA>     25   8.17
leafspots-halo  1                       2    152  49.67
                2                       0    110  35.95
                3                    <NA>     25   8.17
                4                       1     19   6.21
leafspots-marg  1                       0    160  52.29
    

In [100]:
# Target Distribution
target_df

,count,pct
disease,,
alternarialeaf-spot,40,13.07
brown-spot,40,13.07
phytophthora-rot,40,13.07
frog-eye-leaf-spot,40,13.07
brown-stem-rot,20,6.54
anthracnose,20,6.54
downy-mildew,10,3.27
purple-seed-stain,10,3.27
powdery-mildew,10,3.27


## Task Curation

In [101]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [102]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_iid_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
)


splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits.",
    splits=splits,
)

Using Stratified IID splits.


## Export

In [103]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019d0194-75df-7c19-9375-e31b17feec86
1a7cf6471a4a7c5a71a2631a7d51a96daad1a5946d7928b17286f34c3da661d5
